In [1]:
from google.cloud import bigquery
import pandas as pd

project_id = "centered-sight-492614-f0"
json_path =  "/home/khanhdo/Documents/project/bigdata_mining/preprocessing/bigquery.json"
client = bigquery.Client.from_service_account_json(json_path, project = project_id)

/home/khanhdo/anaconda3/envs/spark_lab/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
E0000 00:00:1776856326.354102   42961 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1776856326.354151   42961 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1776856326.354154   42961 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1776856326.354157   

In [2]:
query_transactions = """
SELECT  
    `hash`,
    from_address, 
    to_address, 
    CAST(value AS NUMERIC) / 1e18 AS eth_value,
    block_timestamp, 
    gas_price

FROM `bigquery-public-data.crypto_ethereum.transactions` 
WHERE 
    DATE(block_timestamp) >= "2026-03-21" and DATE(block_timestamp) < "2026-04-01"
    AND receipt_status = 1 
    AND to_address IS NOT NULL
"""
df_transactions = client.query(query_transactions).to_dataframe()
df_transactions.head()

,hash,from_address,to_address,eth_value,block_timestamp,gas_price
0,0xa79ee10e04707324914b825e76fe2329b8fd88850f41...,0x34f5c36dfd35ef66c996eca754bc5e38165f31f4,0xa209d4df0f82767b7993348f3dd09c17709015ed,0E-9,2026-03-26 01:06:11+00:00,49550428
1,0x57aa4bfdcf30247d35696865f77d5f481227a6f42e65...,0xbf7a9a4ada38c23342cc45c7fe87c41e86ac939b,0xbfba6aa165c337793fb59984cce9e644afca5202,1E-9,2026-03-31 23:21:35+00:00,130579312
2,0x51e87296e94e0fa7cb3e882c8cc5a8927e35ac1b3e4f...,0xfa2b8c1f4df2b08b376f936050880176d022ee37,0xb0bb728fac35040313e276ff33b96d257419fe25,1E-9,2026-03-31 23:34:47+00:00,115279765
3,0x5b52ce3bc8216f432aeb157fb30a9b67d72d8ed38a3a...,0xd011adf9ac1e5c1540f8786924e0a67f0f125e8e,0xb47f9648f6b6fdb725d9fdec15d42c90373b6263,1E-9,2026-03-31 23:40:11+00:00,131652304
4,0xbb1031fd16fc2417ec2751435c03ddb6b383416b51bc...,0x113d4e75d64d568cb212d731193650c82077f380,0x6841398a8fe99358bd71a2c0408b40560f5cca63,1E-9,2026-03-31 23:41:59+00:00,128514239


In [3]:
df_transactions.count()

hash               25091193
from_address       25091193
to_address         25091193
eth_value          25091193
block_timestamp    25091193
gas_price          25091193
dtype: int64

In [4]:
df_transactions.to_parquet( "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/transactions_3.parquet",index=False)

In [5]:
query_tokens = """
select 
    address,
    symbol,
    name,
    decimals,
    total_supply
from `bigquery-public-data.crypto_ethereum.tokens`
where symbol IS NOT NULL 
    AND name IS NOT NULL
    AND decimals IS NOT NULL
    AND total_supply IS NOT NULL
    and safe_cast(total_supply AS NUMERIC) > 0
"""
df_tokens = client.query(query_tokens).to_dataframe()
df_tokens.head()

,address,symbol,name,decimals,total_supply
0,0x71ff01d3fb4d002738aa5a64db0d9e5b19f176de,SuperOX,SOX,4,8000000000000
1,0xe2f9af33fa4a43f21125d214452bf79b7226326c,OX,OX Coin,4,180000000000000
2,0x90c9134b62c7d42e4bf97a80f9c37ff1bf4eb52a,USD,USD,4,100000000
3,0x6a6603015654bbd41a50efdb73773107d7ecad9d,USD,USD,8,10000000
4,0xc3db38e3269ffa8cc066d10e9b01d30f04c86a7e,USD,USD,8,100000000


In [6]:
df_tokens.count()


address         174014
symbol          174014
name            174014
decimals        174014
total_supply    174014
dtype: int64

In [7]:
query_token_transfers = """
SELECT
    token_address,
    from_address,  
    to_address,
    value,
    transaction_hash,
    block_timestamp
FROM `bigquery-public-data.crypto_ethereum.token_transfers`
WHERE
    DATE(block_timestamp) >= "2026-03-21" and date(block_timestamp) < "2026-04-01"
    AND value != '0'
    AND to_address IS NOT NULL
    AND to_address != '0x0000000000000000000000000000000000000000' --Loai bo giao dich chet
"""
df_token_transfers = client.query(query_token_transfers).to_dataframe()
df_token_transfers.head()

,token_address,from_address,to_address,value,transaction_hash,block_timestamp
0,0x9e2e5e920ec1b9e37053f9d687c218da5346a453,0x6c89c6c3dfe834152aa05fa5180ff6df2e831b43,0x502cb4759c58875889e55806903566fc9280539e,15300000,0x18365296ae906c5f2b0898c8cd035d44ef55faee98d6...,2026-03-29 16:42:59+00:00
1,0x43baa676ab627b4e9e10928f58531e8096391695,0x43baa676ab627b4e9e10928f58531e8096391695,0x7188dcfdd6279a9a715bb5f1e402add6849e472b,10000000000000000000,0xba44a52388f739f71c586eee0d34bb93b0837b731a13...,2026-03-27 01:42:35+00:00
2,0x43baa676ab627b4e9e10928f58531e8096391695,0x43baa676ab627b4e9e10928f58531e8096391695,0x57fceccd07b5ebcc1bef98703a7bd618d442a275,10000000000000000000,0x7e282d3ba7975c8c7ea5478bf24ccd0fe737d82ba300...,2026-03-27 16:48:23+00:00
3,0x43baa676ab627b4e9e10928f58531e8096391695,0x43baa676ab627b4e9e10928f58531e8096391695,0x9e5584e94b58c9d7a6a082def9ebd389fd769fc5,10000000000000000000,0x67a0e62e6f55f3e9e248b1cb5451efd27e7f20a9697e...,2026-03-27 16:48:23+00:00
4,0x43baa676ab627b4e9e10928f58531e8096391695,0x43baa676ab627b4e9e10928f58531e8096391695,0xebb07ff1a78609f2519dc39fceefafb5a3793175,10000000000000000000,0x7e282d3ba7975c8c7ea5478bf24ccd0fe737d82ba300...,2026-03-27 16:48:23+00:00


In [ ]:
df_token_transfers.to_parquet( "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_3.parquet",index=False)

print("Data has been saved to Parquet files.")

In [8]:
df_token_transfers.count()

token_address       38423484
from_address        38423484
to_address          38423484
value               38423484
transaction_hash    38423484
block_timestamp     38423484
dtype: int64

In [2]:
df_contracts = client.query("""
SELECT 
    address, 
    is_erc20, 
    is_erc721
FROM `bigquery-public-data.crypto_ethereum.contracts`
WHERE 
   date(block_timestamp) >= "2023-01-01" and date(block_timestamp) < "2026-04-01"
""").to_dataframe()

df_contracts.head()

,address,is_erc20,is_erc721
0,0xbede08fbe4d1c72814c35df89f550c175683f019,False,False
1,0xca6ae25d8409feeddf7ad4f74daf80226e289773,False,False
2,0xe99dd8d991be9ff81d31fb699429f174c224ec51,False,False
3,0x74527f97cc8b02bf92f54c51b14b00effb726e04,False,False
4,0x1b197a05c482ef0e79e204982d4b3f58294f619e,False,False


In [3]:
df_contracts.count()

address      42809126
is_erc20     42809126
is_erc721    42809126
dtype: int64

In [4]:
df_contracts.to_parquet( "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/contracts_2.parquet",index=False)